In [2]:
with open("users.xml", "r", encoding="utf-8") as file:
    for i in range(20):
        print(file.readline())

<?xml version='1.0' encoding='utf-8'?>

<data>

  <row>

    <id_user>1238</id_user>

    <timestamp_user_creation>2022-02-01 00:00:00</timestamp_user_creation>

    <username>jeffgolfs</username>

    <follower_count>886</follower_count>

    <following_count>204</following_count>

    <profile_url>https://truthsocial.com/@jeffgolfs</profile_url>

    <finished_follower_scrape>f</finished_follower_scrape>

    <finished_following_scrape>f</finished_following_scrape>

    <finished_truth_scrape>f</finished_truth_scrape>

  </row>

  <row>

    <id_user>50</id_user>

    <timestamp_user_creation>2022-02-01 00:00:00</timestamp_user_creation>

    <username>MizW</username>

    <follower_count>2365</follower_count>

    <following_count>2929</following_count>

    <profile_url>https://truthsocial.com/@MizW</profile_url>



In [4]:
import xml.etree.ElementTree as ET

In [ ]:
#parsing XML.
#Siccome users.xml è molto grande, invece di caricare tutto l’albero in memoria, iniziamo con iterparse, che permette di elaborarlo progressivamente.

In [6]:
for event, elem in ET.iterparse("users.xml", events=("end",)):
    if elem.tag == "row":
        print(elem.tag)
        break

row


In [8]:
#come leggere un singolo record
for child in elem:
    print(child.tag, "=", child.text)

id_user = 1238
timestamp_user_creation = 2022-02-01 00:00:00
username = jeffgolfs
follower_count = 886
following_count = 204
profile_url = https://truthsocial.com/@jeffgolfs
finished_follower_scrape = f
finished_following_scrape = f
finished_truth_scrape = f


In [10]:
#quante <row> ci sono
row_count = 0

for event, elem in ET.iterparse("users.xml", events=("end",)):
    if elem.tag == "row":
        row_count += 1
        elem.clear()

print("Numero di record:", row_count)
#clear() --> Dato che users.xml è grande, dopo che abbiamo contato quella row, non abbiamo più bisogno di conservarne in memoria i figli e il testo.

Numero di record: 487292


In [12]:
#487292 record <row>

In [ ]:
#Ogni record <row> contiene sempre gli stessi 9 elementi figli (id_user, timestamp_user_creation, username, ecc.) oppure esistono record con campi mancanti o aggiuntivi?

In [14]:
min_fields = None
max_fields = 0

for event, elem in ET.iterparse("users.xml", events=("end",)):
    if elem.tag == "row":
        num_fields = len(elem)

        if min_fields is None or num_fields < min_fields:
            min_fields = num_fields

        if num_fields > max_fields:
            max_fields = num_fields

        elem.clear()

print("Minimo numero di campi:", min_fields)
print("Massimo numero di campi:", max_fields)

Minimo numero di campi: 9
Massimo numero di campi: 9


In [16]:
#Ora controlliamo quali sono effettivamente i nomi dei campi in tutte le row. 
reference_fields = None
different_structure = 0

for event, elem in ET.iterparse("users.xml", events=("end",)):
    if elem.tag == "row":
        fields = [child.tag for child in elem] #list comprehension

        if reference_fields is None:
            reference_fields = fields

        elif fields != reference_fields:
            different_structure += 1

        elem.clear()

print("Campi di riferimento:", reference_fields)
print("Record con struttura diversa:", different_structure)

Campi di riferimento: ['id_user', 'timestamp_user_creation', 'username', 'follower_count', 'following_count', 'profile_url', 'finished_follower_scrape', 'finished_following_scrape', 'finished_truth_scrape']
Record con struttura diversa: 0


In [18]:
## Tutti i 487.292 record <row> hanno una struttura uniforme:
# contengono gli stessi 9 elementi figli, nello stesso ordine.

In [20]:
missing_values = {field: 0 for field in reference_fields}

for event, elem in ET.iterparse("users.xml", events=("end",)):
    if elem.tag == "row":

        for child in elem:
            if child.text is None:
                missing_values[child.tag] += 1

        elem.clear()

for field, count in missing_values.items():
    print(field, ":", count)

id_user : 0
timestamp_user_creation : 0
username : 0
follower_count : 0
following_count : 0
profile_url : 0
finished_follower_scrape : 0
finished_following_scrape : 0
finished_truth_scrape : 0


In [22]:
# Non risultano elementi privi di contenuto (child.text is None)
# in nessuno dei 9 campi dei 487.292 record.
# per ora abbiamo controllato solo None. (potrebbero esserci anche spazi vuoti)

In [24]:
#Adesso inizierei dai campi più semplici: i tre campi finished_..., che dalla prima row sembravano contenere "f".

#Vogliamo capire quali valori diversi compaiono realmente nell’intero dataset, senza assumere che siano solo f e t
flag_values = {
    "finished_follower_scrape": set(),
    "finished_following_scrape": set(),
    "finished_truth_scrape": set()
}

for event, elem in ET.iterparse("users.xml", events=("end",)):
    if elem.tag == "row":

        for child in elem:
            if child.tag in flag_values:
                flag_values[child.tag].add(child.text)

        elem.clear()

for field, values in flag_values.items():
    print(field, ":", values)

finished_follower_scrape : {'-1', 'f', 't'}
finished_following_scrape : {'-1', 'f', 't'}
finished_truth_scrape : {'-1', 'f', 't'}


In [26]:
from collections import Counter

flag_counts = {
    "finished_follower_scrape": Counter(),
    "finished_following_scrape": Counter(),
    "finished_truth_scrape": Counter()
}

for event, elem in ET.iterparse("users.xml", events=("end",)):
    if elem.tag == "row":

        for child in elem:
            if child.tag in flag_counts:
                flag_counts[child.tag][child.text] += 1

        elem.clear()

for field, counts in flag_counts.items():
    print(field, ":", counts)

finished_follower_scrape : Counter({'-1': 465337, 'f': 18240, 't': 3715})
finished_following_scrape : Counter({'-1': 465337, 'f': 18531, 't': 3424})
finished_truth_scrape : Counter({'-1': 465337, 'f': 18724, 't': 3231})


In [28]:
#stesso numero di -1

### **Caricamento e analisi della struttura del dataset**

DataFrame Pandas, senza effettuare una conversione in CSV.

In [37]:
import pandas as pd

df_users = pd.read_xml("users.xml", xpath=".//row")

print("Dimensioni del dataset:", df_users.shape)
display(df_users.head())

Dimensioni del dataset: (487292, 9)


,id_user,timestamp_user_creation,username,follower_count,following_count,profile_url,finished_follower_scrape,finished_following_scrape,finished_truth_scrape
0,1238,2022-02-01 00:00:00,jeffgolfs,886,204,https://truthsocial.com/@jeffgolfs,f,f,f
1,50,2022-02-01 00:00:00,MizW,2365,2929,https://truthsocial.com/@MizW,f,f,f
2,6951,2022-02-01 00:00:00,SusanCreech,870,1232,https://truthsocial.com/@SusanCreech,f,f,f
3,9861,2022-02-01 00:00:00,BMF6FeeTDeeP,1086,1654,https://truthsocial.com/@BMF6FeeTDeeP,f,f,f
4,34,2022-02-01 00:00:00,DC_Draino,286146,81,https://truthsocial.com/@DC_Draino,t,t,t


**Commento:**  
Il dataset contiene **487.292 record** e **9 variabili**. Ogni record rappresenta un utente e gli elementi presenti nelle `<row>` del file XML sono stati correttamente organizzati come colonne del DataFrame.

### **Analisi dei tipi delle variabili**



In [43]:
df_users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 487292 entries, 0 to 487291
Data columns (total 9 columns):
 #   Column                     Non-Null Count   Dtype 
---  ------                     --------------   ----- 
 0   id_user                    487292 non-null  int64 
 1   timestamp_user_creation    487292 non-null  object
 2   username                   487292 non-null  object
 3   follower_count             487292 non-null  int64 
 4   following_count            487292 non-null  int64 
 5   profile_url                487292 non-null  object
 6   finished_follower_scrape   487292 non-null  object
 7   finished_following_scrape  487292 non-null  object
 8   finished_truth_scrape      487292 non-null  object
dtypes: int64(3), object(6)
memory usage: 33.5+ MB


**Commento:**  
Il dataset contiene tre variabili riconosciute come numeriche (`id_user`, `follower_count` e `following_count`) e sei variabili di tipo `object`. Tutte le colonne presentano 487.292 valori non nulli, pari al numero totale di record. Non risultano quindi valori mancanti riconosciuti automaticamente da Pandas. Sarà comunque necessario verificare successivamente l'eventuale presenza di valori speciali utilizzati per rappresentare informazioni mancanti o particolari. `timestamp_user_creation`, attualmente di tipo `object`, sarà inoltre analizzata per verificare la possibilità di convertirla in un tipo temporale appropriato.

### **Missing values**


In [48]:
missing_values = df_users.isna().sum()

print(missing_values)

id_user                      0
timestamp_user_creation      0
username                     0
follower_count               0
following_count              0
profile_url                  0
finished_follower_scrape     0
finished_following_scrape    0
finished_truth_scrape        0
dtype: int64


### **Analisi delle variabili relative al completamento dello scraping**

variabili `finished_follower_scrape`, `finished_following_scrape` e `finished_truth_scrape`

In [51]:
scrape_columns = [
    "finished_follower_scrape",
    "finished_following_scrape",
    "finished_truth_scrape"
]

for column in scrape_columns:
    print(f"\n{column}")
    print(df_users[column].value_counts(dropna=False))


finished_follower_scrape
finished_follower_scrape
-1    465337
f      18240
t       3715
Name: count, dtype: int64

finished_following_scrape
finished_following_scrape
-1    465337
f      18531
t       3424
Name: count, dtype: int64

finished_truth_scrape
finished_truth_scrape
-1    465337
f      18724
t       3231
Name: count, dtype: int64


**Commento:**  
Le tre variabili relative al completamento dello scraping assumono ciascuna tre valori distinti: `-1`, `f` e `t`. Il valore `-1` è nettamente predominante e compare 465.337 volte in ciascuna delle tre variabili. I valori `f` e `t` presentano invece frequenze differenti tra i tre campi.

### **Analisi congiunta dei valori `-1` nelle variabili di scraping**



In [55]:
all_minus_one = (
    (df_users["finished_follower_scrape"] == "-1") &
    (df_users["finished_following_scrape"] == "-1") &
    (df_users["finished_truth_scrape"] == "-1")
)

print("Record con -1 in tutti e tre i campi:", all_minus_one.sum())

Record con -1 in tutti e tre i campi: 465337


**Commento:**  
I 465.337 record che presentano il valore `-1` in `finished_follower_scrape` sono gli stessi che presentano `-1` anche in `finished_following_scrape` e `finished_truth_scrape`. 

### **Analisi delle combinazioni dei valori `t` e `f` nelle variabili di scraping**



In [64]:
scrape_combinations = (
    df_users.loc[~all_minus_one, scrape_columns]
    .value_counts()
    .reset_index(name="count")
)

display(scrape_combinations)

,finished_follower_scrape,finished_following_scrape,finished_truth_scrape,count
0,f,f,f,18240
1,t,t,t,3231
2,t,f,f,291
3,t,t,f,193


### **Analisi dell'identificativo degli utenti**

stabilire se `id_user` identifica univocamente ciascun record del dataset.

In [67]:
unique_ids = df_users["id_user"].nunique()
duplicated_ids = df_users["id_user"].duplicated().sum()

print("Numero di record:", len(df_users))
print("ID distinti:", unique_ids)
print("ID duplicati:", duplicated_ids)

Numero di record: 487292
ID distinti: 487292
ID duplicati: 0


**Commento:**  
Il numero di valori distinti di `id_user` coincide con il numero totale di record (487.292) e non sono presenti identificativi duplicati. `id_user` identifica quindi univocamente ciascun record del dataset e può essere utilizzato come chiave identificativa degli utenti.

### **Analisi degli username**


In [73]:
unique_usernames = df_users["username"].nunique()
duplicated_usernames = df_users["username"].duplicated().sum()

print("Numero di record:", len(df_users))
print("Username distinti:", unique_usernames)
print("Username duplicati:", duplicated_usernames)

Numero di record: 487292
Username distinti: 487153
Username duplicati: 139


In [75]:
#vediamo i duplicati
duplicated_username_rows = df_users[
    df_users["username"].duplicated(keep=False) #keep = False per mostrare anche la prima occorrenza 
].sort_values("username")

display(
    duplicated_username_rows[
        ["id_user", "username"]
    ]
)

,id_user,username
119490,119560,-1
433396,433466,-1
414231,414301,-1
412009,412079,-1
348238,348308,-1
...,...,...
445547,445617,willheath
445349,445419,zzzz4u
445347,445417,zzzz4u
445346,445416,zzzz4u


In [79]:
#vediamo la frequenza dei valori coinvolti nelle duplicazioni
duplicated_username_counts = (
    duplicated_username_rows["username"]
    .value_counts()
)

display(duplicated_username_counts)

username
-1                    18
alliechi               8
TiaLou83               8
EllE1223               8
Echo12                 6
                      ..
RobConklin             2
RoxanneWagner          2
SpringRiverFloater     2
DougVark               2
PepperMintPatty47      2
Name: count, Length: 75, dtype: int64

**Commento:**  
La variabile `username` non identifica univocamente i record: sono presenti 487.153 username distinti su 487.292 utenti. 

Questa è una conclusione abbastanza importante: per eventuali join futuri, id_user è affidabile come identificatore univoco; username no.

### **Analisi della data di creazione degli utenti**


In [84]:
creation_datetime = pd.to_datetime(
    df_users["timestamp_user_creation"],
    errors="coerce"
)

invalid_dates = creation_datetime.isna().sum()

print("Date non convertibili:", invalid_dates)

Date non convertibili: 465337


In [86]:
invalid_timestamp_values = df_users.loc[
    creation_datetime.isna(),
    "timestamp_user_creation"
].value_counts(dropna=False)

display(invalid_timestamp_values)

timestamp_user_creation
-1    465337
Name: count, dtype: int64

In [88]:
valid_creation_dates = creation_datetime.dropna()

print("Numero di date valide:", len(valid_creation_dates))
print("Data minima:", valid_creation_dates.min())
print("Data massima:", valid_creation_dates.max())

Numero di date valide: 21955
Data minima: 1969-12-01 00:00:00
Data massima: 2022-12-01 00:00:00


In [90]:
creation_date_counts = valid_creation_dates.value_counts().sort_index()

display(creation_date_counts)

timestamp_user_creation
1969-12-01        2
2021-09-01        5
2021-12-01        3
2022-01-01        6
2022-02-01    16262
2022-03-01     1494
2022-04-01     1048
2022-05-01     2211
2022-06-01      481
2022-07-01      181
2022-08-01      162
2022-09-01       59
2022-10-01       34
2022-11-01        6
2022-12-01        1
Name: count, dtype: int64

**Commento:**  
La variabile `timestamp_user_creation` contiene 465.337 valori `-1`, che non possono essere interpretati come date, e 21.955 valori convertibili correttamente in formato temporale. Le date valide sono comprese tra dicembre 1969 e dicembre 2022 e risultano sempre associate al primo giorno del mese. 

### **Analisi di `follower_count` e `following_count`**



In [96]:
count_columns = ["follower_count", "following_count"]

display(
    df_users[count_columns]
    .describe()
    .round(2)
    .style.format("{:,.2f}")
)

,follower_count,following_count
count,"487,292.00","487,292.00"
mean,328.48,150.32
std,"12,051.40","1,332.52"
min,-1.00,-1.00
25%,-1.00,-1.00
50%,-1.00,-1.00
75%,-1.00,-1.00
max,"4,027,829.00","99,514.00"


### **Analisi dei conteggi effettivi di follower e following**


In [101]:
# Analizziamo separatamente i conteggi effettivi, escludendo il valore speciale -1

valid_follower = df_users.loc[
    df_users["follower_count"] != -1,
    "follower_count"
]

valid_following = df_users.loc[
    df_users["following_count"] != -1,
    "following_count"
]

print("Follower count:")
display(valid_follower.describe().round(2))

print("\nFollowing count:")
display(valid_following.describe().round(2))

Follower count:


count      21955.00
mean        7311.81
std        56325.80
min            0.00
25%          414.00
50%         1929.00
75%         5139.50
max      4027829.00
Name: follower_count, dtype: float64


Following count:


count    21955.0
mean      3357.5
std       5351.6
min          0.0
25%        150.0
50%       1540.0
75%       5389.0
max      99514.0
Name: following_count, dtype: float64

### **Analisi dei valori più elevati di follower e following**


In [108]:
# Visualizziamo i 10 utenti con il maggior numero di follower

top_followers = (
    df_users.loc[
        df_users["follower_count"] != -1,
        ["id_user", "username", "follower_count", "following_count"]
    ]
    .sort_values("follower_count", ascending=False)
    .head(10)
)

display(top_followers)

,id_user,username,follower_count,following_count
19118,27,realDonaldTrump,4027829,3
564,300,DonaldJTrumpJr,2524434,227
246,160,truthsocial,2375491,2
85,105,dbongino,1955706,249
175,211,BabylonBee,1809193,1
78,26,EricTrump,1727967,178
284,145,NewsMax,1511293,48
288,322,DineshDSouza,1491112,301
863,103,seanhannity,1288592,6
129,147,rumble,1142497,58


In [110]:
# Visualizziamo i 10 utenti con il maggior numero di account seguiti

top_following = (
    df_users.loc[
        df_users["following_count"] != -1,
        ["id_user", "username", "follower_count", "following_count"]
    ]
    .sort_values("following_count", ascending=False)
    .head(10)
)

display(top_following)

,id_user,username,follower_count,following_count
1012,922,Blondies,97618,99514
2164,2189,TheExecutioner,82661,88242
2695,2649,theworldupdater,78095,83756
9926,9037,cyberhunter,80400,82711
2167,2192,luvSB1,78695,80633
2166,2191,OksanaTrump,79703,80533
2163,2188,JordynTrump,77202,77488
2328,2357,Shadowknight,64696,71300
8795,8425,DavidYeshua,68782,70979
2170,2195,Sheerglee,73218,70930


### **Analisi degli URL dei profili**


In [114]:
# Verifichiamo l'unicità degli URL dei profili

unique_urls = df_users["profile_url"].nunique()
duplicated_urls = df_users["profile_url"].duplicated().sum()

print("Numero di record:", len(df_users))
print("URL distinti:", unique_urls)
print("URL duplicati:", duplicated_urls)

Numero di record: 487292
URL distinti: 487150
URL duplicati: 142


In [116]:
# Individuiamo tutti i record coinvolti nella duplicazione degli URL

duplicated_url_rows = df_users[
    df_users["profile_url"].duplicated(keep=False)
].sort_values("profile_url")

display(
    duplicated_url_rows[
        ["id_user", "username", "profile_url"]
    ]
)

,id_user,username,profile_url
445577,445647,1HERO,https://truthsocial.com/@1HERO
445464,445534,1HERO,https://truthsocial.com/@1HERO
445331,445401,4Freedomrider,https://truthsocial.com/@4Freedomrider
445332,445402,4Freedomrider,https://truthsocial.com/@4Freedomrider
445190,445260,76BORN,https://truthsocial.com/@76BORN
...,...,...,...
445516,445586,willheath,https://truthsocial.com/@willheath
445349,445419,zzzz4u,https://truthsocial.com/@zzzz4u
445348,445418,zzzz4u,https://truthsocial.com/@zzzz4u
445346,445416,zzzz4u,https://truthsocial.com/@zzzz4u


nota la relazione con username

In [118]:
# Contiamo la frequenza degli URL coinvolti nelle duplicazioni

duplicated_url_counts = (
    duplicated_url_rows["profile_url"]
    .value_counts()
)

display(duplicated_url_counts)

profile_url
https://truthsocial.com/@nan                    18
https://truthsocial.com/@alliechi                8
https://truthsocial.com/@TiaLou83                8
https://truthsocial.com/@EllE1223                8
https://truthsocial.com/@flamingconservative     6
                                                ..
https://truthsocial.com/@QueenCate               2
https://truthsocial.com/@4Freedomrider           2
https://truthsocial.com/@RoxanneWagner           2
https://truthsocial.com/@SpringRiverFloater      2
https://truthsocial.com/@RobConklin              2
Name: count, Length: 78, dtype: int64

In [120]:
# Analizziamo i record associati all'URL che termina con @nan

nan_url_rows = df_users[
    df_users["profile_url"] == "https://truthsocial.com/@nan"
]

display(
    nan_url_rows[
        ["id_user", "username", "profile_url"]
    ]
)

,id_user,username,profile_url
119490,119560,-1,https://truthsocial.com/@nan
127811,127881,-1,https://truthsocial.com/@nan
178390,178460,-1,https://truthsocial.com/@nan
210208,210278,-1,https://truthsocial.com/@nan
275812,275882,-1,https://truthsocial.com/@nan
277411,277481,-1,https://truthsocial.com/@nan
280255,280325,-1,https://truthsocial.com/@nan
284401,284471,-1,https://truthsocial.com/@nan
319040,319110,-1,https://truthsocial.com/@nan
335832,335902,-1,https://truthsocial.com/@nan


### **Verifica della relazione tra `username` e `profile_url`**


In [126]:
# Verifichiamo la corrispondenza tra username e profile_url

valid_username = df_users["username"] != "-1"

expected_url = (
    "https://truthsocial.com/@" +
    df_users.loc[valid_username, "username"].astype(str)
)

url_matches = (
    df_users.loc[valid_username, "profile_url"] == expected_url
)

print("Record controllati:", url_matches.size)
print("URL coerenti con username:", url_matches.sum())
print("URL non coerenti con username:", (~url_matches).sum())

Record controllati: 487274
URL coerenti con username: 487271
URL non coerenti con username: 3


In [128]:
# Visualizziamo i record in cui profile_url non corrisponde allo username

inconsistent_urls = df_users.loc[
    valid_username & ~url_matches.reindex(df_users.index, fill_value=False),
    ["id_user", "username", "profile_url"]
]

display(inconsistent_urls)

,id_user,username,profile_url
6239,7804,WSJPolitics,https://truthsocial.com/@cpac
7040,6384,bean8920,https://truthsocial.com/@dani
7250,6582,SiydenGalanter,https://truthsocial.com/@lovelynn


### **Analisi dei record duplicati**


In [132]:
# Verifichiamo la presenza di record completamente duplicati

duplicated_rows = df_users.duplicated().sum()

print("Record completamente duplicati:", duplicated_rows)

Record completamente duplicati: 0
